# Task 2: Neural Network – MNIST Handwritten Digit Classification

---

## Objective

The objective of this notebook is to build, train, and evaluate a **simple feed-forward neural network (ANN)** to classify handwritten digits (0–9) from the **MNIST dataset** using **TensorFlow/Keras**.

We will:
1. Understand the MNIST dataset.
2. Preprocess the data (normalization, flattening).
3. Build a baseline fully connected neural network.
4. Explain activation functions (ReLU and Softmax).
5. Train the model and visualize training curves.
6. Evaluate the model using accuracy, confusion matrix, precision, recall, and F1-score.
7. Perform a controlled experiment by modifying the network architecture.

---

## Import Libraries

We begin by importing all the libraries we will need throughout this notebook:

- **TensorFlow / Keras** – for building and training the neural network.
- **NumPy** – for numerical operations on arrays.
- **Matplotlib** – for plotting graphs and displaying images.
- **Seaborn** – for creating attractive confusion matrix heatmaps.
- **scikit-learn** – for evaluation metrics (confusion matrix, classification report).

In [ ]:
# ============================================================
# Import all required libraries
# ============================================================

import numpy as np                          # Numerical operations
import matplotlib.pyplot as plt             # Plotting and visualization
import seaborn as sns                       # Heatmap for confusion matrix

import tensorflow as tf                     # Deep learning framework
from tensorflow import keras                # High-level API for building models
from tensorflow.keras import layers         # Neural network layers
from tensorflow.keras.datasets import mnist # Built-in MNIST dataset

from sklearn.metrics import confusion_matrix, classification_report  # Evaluation

# ============================================================
# Set random seeds for reproducibility
# ============================================================
# Setting seeds ensures that the results are consistent across runs.
# Note: Results may still vary slightly due to GPU non-determinism.

np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print("All libraries imported successfully.")

---

## Part A – Dataset Understanding

### What is the MNIST Dataset?

The **MNIST** (Modified National Institute of Standards and Technology) dataset is one of the most well-known benchmark datasets in machine learning. It consists of **grayscale images of handwritten digits** from 0 to 9.

**Key facts about MNIST:**

| Property | Value |
|---|---|
| **Content** | Handwritten digits (0, 1, 2, 3, 4, 5, 6, 7, 8, 9) |
| **Image dimensions** | 28 × 28 pixels |
| **Color** | Grayscale (single channel) |
| **Pixel value range** | 0 (black) to 255 (white) |
| **Number of classes** | 10 (one for each digit 0–9) |
| **Training images** | 60,000 |
| **Testing images** | 10,000 |
| **Total images** | 70,000 |

**What the labels represent:**  
Each image has a corresponding **label** (an integer from 0 to 9) indicating which digit the image contains. For example, an image of a handwritten "3" has the label `3`.

---

### Load and Explore the Dataset

In [ ]:
# ============================================================
# Load the MNIST dataset from Keras
# ============================================================
# Keras provides MNIST as a built-in dataset.
# It automatically splits the data into training and testing sets.

(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = mnist.load_data()

# ============================================================
# Display dataset shapes
# ============================================================

print("===== MNIST Dataset Shapes =====")
print(f"Training images shape : {x_train_raw.shape}")   # (60000, 28, 28)
print(f"Training labels shape : {y_train_raw.shape}")   # (60000,)
print(f"Testing images shape  : {x_test_raw.shape}")    # (10000, 28, 28)
print(f"Testing labels shape  : {y_test_raw.shape}")    # (10000,)
print()
print(f"Number of training samples : {x_train_raw.shape[0]}")
print(f"Number of testing samples  : {x_test_raw.shape[0]}")
print(f"Image dimensions           : {x_train_raw.shape[1]} x {x_train_raw.shape[2]} pixels")
print(f"Pixel value range          : {x_train_raw.min()} to {x_train_raw.max()}")
print(f"Number of classes          : {len(np.unique(y_train_raw))}")
print(f"Unique labels              : {np.unique(y_train_raw)}")
print(f"Data type                  : {x_train_raw.dtype}")

### Display Sample Handwritten Digit Images

Let us visualize a grid of sample images from the training set along with their corresponding labels. This helps us understand what the data looks like.

In [ ]:
# ============================================================
# Display a 3x5 grid of sample images with their labels
# ============================================================

fig, axes = plt.subplots(3, 5, figsize=(12, 8))
fig.suptitle("Sample Handwritten Digits from MNIST Training Set", fontsize=16, fontweight='bold')

for i, ax in enumerate(axes.flat):
    ax.imshow(x_train_raw[i], cmap='gray')          # Display image in grayscale
    ax.set_title(f"Label: {y_train_raw[i]}", fontsize=12)
    ax.axis('off')                                    # Hide axis ticks

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Display the distribution of digit classes in the training set
# ============================================================

unique, counts = np.unique(y_train_raw, return_counts=True)

plt.figure(figsize=(8, 5))
plt.bar(unique, counts, color='steelblue', edgecolor='black')
plt.xlabel("Digit Class", fontsize=12)
plt.ylabel("Number of Samples", fontsize=12)
plt.title("Distribution of Digit Classes in Training Set", fontsize=14, fontweight='bold')
plt.xticks(unique)
for i, count in enumerate(counts):
    plt.text(i, count + 100, str(count), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

---

## Part B – Data Preprocessing

Before feeding the data into our neural network, we need to **preprocess** it. The preprocessing steps are:

### 1. Normalize Pixel Values (0–255 → 0–1)

**Why normalize?**  
- Raw pixel values range from 0 to 255. Large input values can cause the neural network to learn slowly or become unstable.
- Dividing by 255 scales all values to the range **[0, 1]**, which helps the optimizer (like Adam) converge faster.
- Normalization ensures that all features (pixels) contribute equally during training.

### 2. Flatten Images (28×28 → 784)

**Why flatten?**  
- Our neural network is a **fully connected (dense) network**, not a CNN.
- A dense layer expects a **1D vector** as input, not a 2D grid.
- Each 28×28 image is reshaped into a single vector of **784 values** (28 × 28 = 784).
- This means each pixel becomes an individual input feature to the network.

### 3. Training, Validation, and Test Sets

| Set | Purpose | Source |
|---|---|---|
| **Training set** | Used to train the model (update weights) | First 50,000 images from the original training data |
| **Validation set** | Used to monitor performance during training and detect overfitting | Last 10,000 images from the original training data |
| **Test set** | Used for final evaluation after training is complete | The 10,000 images provided separately |

**Key principle:** The test set is **never** used during training. It is only used at the very end to evaluate how well the model generalizes to unseen data.

In [ ]:
# ============================================================
# Step 1: Normalize pixel values from [0, 255] to [0, 1]
# ============================================================
# We convert to float32 first, then divide by 255.

x_train_normalized = x_train_raw.astype('float32') / 255.0
x_test_normalized  = x_test_raw.astype('float32') / 255.0

print(f"After normalization:")
print(f"  Training pixel range: {x_train_normalized.min():.1f} to {x_train_normalized.max():.1f}")
print(f"  Testing pixel range : {x_test_normalized.min():.1f} to {x_test_normalized.max():.1f}")

In [ ]:
# ============================================================
# Step 2: Flatten images from 28x28 to 784
# ============================================================
# reshape(-1, 784) means: keep all samples, flatten each to 784.

x_train_flat = x_train_normalized.reshape(-1, 784)   # (60000, 784)
x_test_flat  = x_test_normalized.reshape(-1, 784)    # (10000, 784)

print(f"After flattening:")
print(f"  Training data shape : {x_train_flat.shape}")
print(f"  Testing data shape  : {x_test_flat.shape}")

In [ ]:
# ============================================================
# Step 3: Create a validation set from the training data
# ============================================================
# We split the 60,000 training images:
#   - First 50,000 → training
#   - Last  10,000 → validation

x_val   = x_train_flat[50000:]     # Validation images (10,000)
y_val   = y_train_raw[50000:]      # Validation labels

x_train = x_train_flat[:50000]     # Training images (50,000)
y_train = y_train_raw[:50000]      # Training labels

x_test  = x_test_flat              # Test images (10,000)
y_test  = y_test_raw               # Test labels

print("===== Final Data Splits =====")
print(f"Training set   : {x_train.shape[0]} samples, shape {x_train.shape}")
print(f"Validation set : {x_val.shape[0]} samples, shape {x_val.shape}")
print(f"Test set       : {x_test.shape[0]} samples, shape {x_test.shape}")

---

## Part C – Build the Neural Network

We will build a **simple feed-forward neural network** (also called an Artificial Neural Network or ANN) using the Keras Sequential API.

### Architecture: 784 → 128 → 10

| Layer | Type | Neurons | Activation | Purpose |
|---|---|---|---|---|
| **Input** | Input | 784 | – | Receives the flattened 28×28 image as a vector of 784 pixel values |
| **Hidden Layer** | Dense (fully connected) | 128 | ReLU | Learns patterns and features from the input data |
| **Output Layer** | Dense (fully connected) | 10 | Softmax | Produces a probability distribution over 10 digit classes |

### Key Components Explained

- **Input Layer (784 features):** Each pixel in the flattened image is treated as an independent input feature. So a 28×28 image becomes 784 input values.

- **Dense (Fully Connected) Layer:** Every neuron in one layer is connected to every neuron in the next layer. Each connection has a learnable weight.

- **128 Neurons in the Hidden Layer:** This is a reasonable number for a simple model. These neurons learn intermediate representations (patterns) from the pixel data.

- **ReLU Activation:** Applied to the hidden layer. It introduces non-linearity, allowing the network to learn complex patterns. (Explained in detail in Part D.)

- **Output Layer (10 neurons):** One neuron per digit class (0–9). Each neuron outputs a value that gets converted into a probability.

- **Softmax Activation:** Applied to the output layer. It converts raw output scores into probabilities that sum to 1. (Explained in detail in Part D.)

### Compilation

- **Optimizer: Adam** – An adaptive learning rate optimizer that is efficient and works well in most cases.
- **Loss: Sparse Categorical Crossentropy** – The standard loss function for multi-class classification when labels are integers (not one-hot encoded).
- **Metric: Accuracy** – The percentage of correctly classified images.

In [ ]:
# ============================================================
# Build the Baseline Neural Network Model
# Architecture: 784 → 128 (ReLU) → 10 (Softmax)
# ============================================================

baseline_model = keras.Sequential([
    # Hidden Layer: 128 neurons, ReLU activation
    # input_shape=(784,) tells Keras that each input sample is a vector of 784 values
    layers.Dense(128, activation='relu', input_shape=(784,)),
    
    # Output Layer: 10 neurons (one per digit class), Softmax activation
    # Softmax converts the outputs into probabilities that sum to 1
    layers.Dense(10, activation='softmax')
])

# ============================================================
# Compile the model
# ============================================================

baseline_model.compile(
    optimizer='adam',                              # Adam optimizer
    loss='sparse_categorical_crossentropy',        # Loss for integer labels
    metrics=['accuracy']                           # Track accuracy
)

# ============================================================
# Display model summary
# ============================================================

print("===== Baseline Model Summary =====")
baseline_model.summary()

**Understanding the model summary:**

- The first Dense layer has **100,480 parameters**: (784 inputs × 128 neurons) + 128 biases = 100,480.
- The second Dense layer has **1,290 parameters**: (128 inputs × 10 neurons) + 10 biases = 1,290.
- **Total trainable parameters: 101,770** — these are the weights and biases that the model learns during training.

---

## Part D – Activation Functions

### What is an Activation Function?

An **activation function** is a mathematical function applied to the output of each neuron in a neural network. It determines whether a neuron should be "activated" (fire) or not.

### Why are Activation Functions Needed?

Without activation functions, a neural network — no matter how many layers it has — would behave like a **single linear transformation**. This is because stacking linear operations just produces another linear operation.

Activation functions introduce **non-linearity**, which allows the network to learn complex, non-linear relationships in the data (such as recognizing curves, edges, and shapes in handwritten digits).

---

### ReLU (Rectified Linear Unit)

$$\text{ReLU}(x) = \max(0, x)$$

- If the input is **positive**, ReLU passes it through unchanged.
- If the input is **negative**, ReLU outputs **zero**.

**Example:**
- ReLU(5) = 5
- ReLU(-3) = 0
- ReLU(0) = 0

**Why ReLU is commonly used in hidden layers:**
1. **Simple and fast** to compute — just a comparison with zero.
2. **Reduces the vanishing gradient problem** — unlike sigmoid, the gradient of ReLU for positive values is always 1, so the network can learn efficiently even with many layers.
3. **Sparsity** — by outputting zero for negative inputs, ReLU creates sparse activations, which can make the network more efficient.

---

### Softmax

$$\text{Softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}$$

where $z_i$ is the raw output of neuron $i$, and $K$ is the total number of classes.

**What Softmax does:**
1. Takes the **10 raw output values** from the output layer.
2. Applies the exponential function to each value (making them all positive).
3. Divides each by the **sum of all exponentials**, so the results add up to 1.
4. The result is a **probability distribution** over 10 classes.

**Example:**  
If the output layer produces raw scores `[2.0, 1.0, 0.1, ..., 0.5]`, Softmax converts them into probabilities like `[0.65, 0.12, 0.01, ..., 0.03]`. The class with the highest probability is the model's prediction.

**Why Softmax is suitable for MNIST:**
- MNIST is a **multi-class classification** problem (10 classes).
- Softmax ensures the model outputs a valid probability for each class.
- The predicted digit is the class with the **highest probability**.
- It works naturally with the **categorical crossentropy** loss function.

In [ ]:
# ============================================================
# Visual demonstration of ReLU and Softmax
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- ReLU Plot ---
x_vals = np.linspace(-5, 5, 200)
relu_vals = np.maximum(0, x_vals)  # ReLU(x) = max(0, x)

axes[0].plot(x_vals, relu_vals, color='blue', linewidth=2)
axes[0].axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
axes[0].axvline(x=0, color='gray', linestyle='--', linewidth=0.5)
axes[0].set_title('ReLU Activation Function', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Input (x)', fontsize=12)
axes[0].set_ylabel('Output: max(0, x)', fontsize=12)
axes[0].set_xlim(-5, 5)
axes[0].set_ylim(-1, 5)
axes[0].grid(True, alpha=0.3)

# --- Softmax Demonstration ---
# Simulate raw output scores for 10 classes
raw_scores = np.array([2.0, 1.0, 0.1, 0.5, 3.5, 0.2, 0.8, 0.3, 1.5, 0.6])
softmax_probs = np.exp(raw_scores) / np.sum(np.exp(raw_scores))  # Softmax formula

colors = ['steelblue'] * 10
colors[np.argmax(softmax_probs)] = 'red'  # Highlight the predicted class

axes[1].bar(range(10), softmax_probs, color=colors, edgecolor='black')
axes[1].set_title('Softmax Output (Probabilities)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Digit Class', fontsize=12)
axes[1].set_ylabel('Probability', fontsize=12)
axes[1].set_xticks(range(10))
axes[1].set_ylim(0, max(softmax_probs) + 0.05)

# Add probability values on top of bars
for i, prob in enumerate(softmax_probs):
    axes[1].text(i, prob + 0.01, f"{prob:.3f}", ha='center', fontsize=8)

plt.tight_layout()
plt.show()

print(f"Raw output scores : {raw_scores}")
print(f"After Softmax     : {np.round(softmax_probs, 4)}")
print(f"Sum of Softmax    : {softmax_probs.sum():.4f}  (always equals 1.0)")
print(f"Predicted class   : {np.argmax(softmax_probs)} (highest probability)")

---

## Part E – Training

We now train the baseline model on the training data. During training:
- The model sees the training data and updates its weights to minimize the loss.
- After each epoch (one full pass through the training data), we evaluate the model on the **validation set** to monitor generalization.

**Training parameters:**
- **Epochs: 15** – The model will pass through the entire training dataset 15 times.
- **Batch size: 32** – The model processes 32 images at a time before updating weights.
- **Validation data** – Used to track how well the model generalizes after each epoch.

In [ ]:
# ============================================================
# Train the Baseline Model
# ============================================================

print("Training the Baseline Model (784 → 128 → 10)...")
print("="*50)

baseline_history = baseline_model.fit(
    x_train, y_train,              # Training data and labels
    epochs=15,                      # Number of training epochs
    batch_size=32,                  # Number of samples per batch
    validation_data=(x_val, y_val), # Validation data for monitoring
    verbose=1                       # Show training progress
)

print("\nTraining complete!")

### Training Curves

We plot the **accuracy** and **loss** curves for both training and validation sets across epochs.

- **Accuracy curves:** If training accuracy is much higher than validation accuracy, the model may be **overfitting** (memorizing training data instead of learning general patterns).
- **Loss curves:** Training and validation loss should both decrease. If validation loss starts increasing while training loss continues to decrease, this indicates **overfitting**.

In [ ]:
# ============================================================
# Plot Training and Validation Accuracy & Loss
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Accuracy Plot ---
axes[0].plot(baseline_history.history['accuracy'], label='Training Accuracy', marker='o', markersize=4)
axes[0].plot(baseline_history.history['val_accuracy'], label='Validation Accuracy', marker='s', markersize=4)
axes[0].set_title('Baseline Model: Accuracy over Epochs', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# --- Loss Plot ---
axes[1].plot(baseline_history.history['loss'], label='Training Loss', marker='o', markersize=4)
axes[1].plot(baseline_history.history['val_loss'], label='Validation Loss', marker='s', markersize=4)
axes[1].set_title('Baseline Model: Loss over Epochs', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final epoch values
print(f"\nFinal Training Accuracy   : {baseline_history.history['accuracy'][-1]:.4f}")
print(f"Final Validation Accuracy : {baseline_history.history['val_accuracy'][-1]:.4f}")
print(f"Final Training Loss       : {baseline_history.history['loss'][-1]:.4f}")
print(f"Final Validation Loss     : {baseline_history.history['val_loss'][-1]:.4f}")

**Interpreting the curves:**

- Both training and validation accuracy should increase across epochs, indicating the model is learning.
- Both training and validation loss should decrease, indicating the model is becoming more confident in its predictions.
- If there is a noticeable gap between training and validation curves, it may suggest some degree of overfitting. A small gap is normal and acceptable for a simple model.

---

## Part F – Evaluation

We now evaluate the trained baseline model on the **test set** — data the model has never seen during training.

### Evaluation Metrics:

- **Test Accuracy:** Percentage of test images correctly classified.
- **Test Loss:** How well the model's predicted probabilities match the true labels.
- **Confusion Matrix:** A table showing how many images of each digit were classified as each digit.
- **Precision:** Of all images predicted as a certain digit, how many actually were that digit?
- **Recall:** Of all actual images of a certain digit, how many did the model correctly identify?
- **F1-Score:** The harmonic mean of precision and recall — a balanced measure.

In [ ]:
# ============================================================
# Evaluate the Baseline Model on the Test Set
# ============================================================

baseline_test_loss, baseline_test_accuracy = baseline_model.evaluate(x_test, y_test, verbose=0)

print("===== Baseline Model – Test Set Evaluation =====")
print(f"Test Accuracy : {baseline_test_accuracy:.4f} ({baseline_test_accuracy*100:.2f}%)")
print(f"Test Loss     : {baseline_test_loss:.4f}")

In [ ]:
# ============================================================
# Generate Predictions on the Test Set
# ============================================================

# The model outputs probabilities for each of the 10 classes
baseline_predictions_proba = baseline_model.predict(x_test, verbose=0)

# Get the predicted class (digit with the highest probability)
baseline_predictions = np.argmax(baseline_predictions_proba, axis=1)

print(f"Predictions shape: {baseline_predictions.shape}")
print(f"First 10 predictions : {baseline_predictions[:10]}")
print(f"First 10 true labels : {y_test[:10]}")

### Confusion Matrix

A **confusion matrix** is a table that shows how many images from each true class were classified into each predicted class.

- **Diagonal values** (top-left to bottom-right) represent **correct predictions**.
- **Off-diagonal values** represent **misclassifications** — where the model confused one digit for another.
- A good model will have high values on the diagonal and low (or zero) values off the diagonal.

In [ ]:
# ============================================================
# Create and Display the Confusion Matrix
# ============================================================

# Compute the confusion matrix
cm = confusion_matrix(y_test, baseline_predictions)

# Plot as a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,         # Show numbers in each cell
    fmt='d',            # Format as integers
    cmap='Blues',        # Color scheme
    xticklabels=range(10),
    yticklabels=range(10),
    linewidths=0.5,
    linecolor='gray'
)
plt.title('Baseline Model – Confusion Matrix', fontsize=16, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=13)
plt.ylabel('True Label', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Identify Most Confused Digit Pairs
# ============================================================

# Zero out the diagonal (correct predictions) to find the largest off-diagonal values
cm_off_diag = cm.copy()
np.fill_diagonal(cm_off_diag, 0)

# Find the top 5 most confused pairs
print("===== Most Confused Digit Pairs (Baseline Model) =====")
print(f"{'True Digit':<12} {'Predicted As':<14} {'Count'}")
print("-" * 40)

# Get indices of the largest off-diagonal values
flat_indices = np.argsort(cm_off_diag.flatten())[::-1][:5]
for idx in flat_indices:
    true_digit = idx // 10
    pred_digit = idx % 10
    count = cm_off_diag[true_digit, pred_digit]
    if count > 0:
        print(f"{true_digit:<12} {pred_digit:<14} {count}")

In [ ]:
# ============================================================
# Classification Report: Precision, Recall, F1-Score
# ============================================================

print("===== Baseline Model – Classification Report =====")
print()
print(classification_report(
    y_test,
    baseline_predictions,
    target_names=[f"Digit {i}" for i in range(10)]
))

### Display Sample Predictions

Let us visualize some test images along with their actual and predicted labels. We will show both **correctly classified** and **incorrectly classified** examples.

In [ ]:
# ============================================================
# Display Correctly Classified Examples
# ============================================================

correct_indices = np.where(baseline_predictions == y_test)[0]
incorrect_indices = np.where(baseline_predictions != y_test)[0]

print(f"Correctly classified   : {len(correct_indices)} / {len(y_test)}")
print(f"Incorrectly classified : {len(incorrect_indices)} / {len(y_test)}")

# --- Show 10 correctly classified images ---
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle('Correctly Classified Examples', fontsize=16, fontweight='bold', color='green')

for i, ax in enumerate(axes.flat):
    idx = correct_indices[i]
    ax.imshow(x_test[idx].reshape(28, 28), cmap='gray')
    ax.set_title(f"True: {y_test[idx]} | Pred: {baseline_predictions[idx]}", fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Display Incorrectly Classified Examples (if any)
# ============================================================

if len(incorrect_indices) > 0:
    num_to_show = min(10, len(incorrect_indices))
    cols = min(5, num_to_show)
    rows = (num_to_show + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(14, 3 * rows))
    fig.suptitle('Incorrectly Classified Examples', fontsize=16, fontweight='bold', color='red')

    # Handle the case where axes might be 1D or a single axis
    if num_to_show == 1:
        axes = np.array([axes])
    axes_flat = axes.flatten()

    for i in range(len(axes_flat)):
        if i < num_to_show:
            idx = incorrect_indices[i]
            axes_flat[i].imshow(x_test[idx].reshape(28, 28), cmap='gray')
            axes_flat[i].set_title(
                f"True: {y_test[idx]} | Pred: {baseline_predictions[idx]}",
                fontsize=10, color='red'
            )
        axes_flat[i].axis('off')

    plt.tight_layout()
    plt.show()
else:
    print("No incorrectly classified examples found!")

---

## Part G – Experimentation

### Controlled Experiment: Increasing Hidden Layer Neurons

To understand how the network architecture affects performance, we perform a **controlled experiment** where we change **only one thing** — the number of neurons in the hidden layer — and keep everything else the same.

| Aspect | Baseline Model | Modified Model |
|---|---|---|
| **Architecture** | 784 → **128** → 10 | 784 → **256** → 10 |
| **Hidden neurons** | 128 | 256 |
| **Optimizer** | Adam | Adam |
| **Loss** | Sparse Categorical Crossentropy | Sparse Categorical Crossentropy |
| **Epochs** | 15 | 15 |
| **Batch size** | 32 | 32 |
| **Dataset** | Same MNIST | Same MNIST |

**Hypothesis:** Increasing the number of hidden neurons from 128 to 256 gives the model more capacity to learn patterns, which may improve accuracy. However, a larger model also has more parameters, which could lead to overfitting if the model memorizes the training data.

In [ ]:
# ============================================================
# Build the Modified Model: 784 → 256 → 10
# ============================================================

# Reset random seeds for a fair comparison
np.random.seed(42)
tf.random.set_seed(42)

modified_model = keras.Sequential([
    # Hidden Layer: 256 neurons (increased from 128), ReLU activation
    layers.Dense(256, activation='relu', input_shape=(784,)),
    
    # Output Layer: 10 neurons, Softmax activation
    layers.Dense(10, activation='softmax')
])

# Compile with the same settings as the baseline
modified_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("===== Modified Model Summary =====")
modified_model.summary()

In [ ]:
# ============================================================
# Train the Modified Model
# ============================================================

print("Training the Modified Model (784 → 256 → 10)...")
print("="*50)

modified_history = modified_model.fit(
    x_train, y_train,
    epochs=15,
    batch_size=32,
    validation_data=(x_val, y_val),
    verbose=1
)

print("\nTraining complete!")

In [ ]:
# ============================================================
# Evaluate the Modified Model on the Test Set
# ============================================================

modified_test_loss, modified_test_accuracy = modified_model.evaluate(x_test, y_test, verbose=0)

print("===== Modified Model – Test Set Evaluation =====")
print(f"Test Accuracy : {modified_test_accuracy:.4f} ({modified_test_accuracy*100:.2f}%)")
print(f"Test Loss     : {modified_test_loss:.4f}")

---

## Model Comparison

We now compare the baseline (128 hidden neurons) and modified (256 hidden neurons) models side by side.

In [ ]:
# ============================================================
# Comparison Table
# ============================================================

print("=" * 65)
print(f"{'Metric':<30} {'Baseline (128)':<18} {'Modified (256)'}")
print("=" * 65)
print(f"{'Hidden Neurons':<30} {'128':<18} {'256'}")
print(f"{'Total Parameters':<30} {baseline_model.count_params():<18} {modified_model.count_params()}")
print(f"{'Training Accuracy':<30} {baseline_history.history['accuracy'][-1]:<18.4f} {modified_history.history['accuracy'][-1]:.4f}")
print(f"{'Validation Accuracy':<30} {baseline_history.history['val_accuracy'][-1]:<18.4f} {modified_history.history['val_accuracy'][-1]:.4f}")
print(f"{'Validation Loss':<30} {baseline_history.history['val_loss'][-1]:<18.4f} {modified_history.history['val_loss'][-1]:.4f}")
print(f"{'Test Accuracy':<30} {baseline_test_accuracy:<18.4f} {modified_test_accuracy:.4f}")
print(f"{'Test Loss':<30} {baseline_test_loss:<18.4f} {modified_test_loss:.4f}")
print("=" * 65)

In [ ]:
# ============================================================
# Comparison Visualization: Training Curves Side by Side
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Accuracy Comparison ---
axes[0].plot(baseline_history.history['val_accuracy'], label='Baseline (128) – Val Acc', marker='o', markersize=4)
axes[0].plot(modified_history.history['val_accuracy'], label='Modified (256) – Val Acc', marker='s', markersize=4)
axes[0].set_title('Validation Accuracy Comparison', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Validation Accuracy', fontsize=12)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# --- Loss Comparison ---
axes[1].plot(baseline_history.history['val_loss'], label='Baseline (128) – Val Loss', marker='o', markersize=4)
axes[1].plot(modified_history.history['val_loss'], label='Modified (256) – Val Loss', marker='s', markersize=4)
axes[1].set_title('Validation Loss Comparison', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Validation Loss', fontsize=12)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Bar Chart: Test Accuracy Comparison
# ============================================================

models = ['Baseline\n(128 neurons)', 'Modified\n(256 neurons)']
test_accuracies = [baseline_test_accuracy, modified_test_accuracy]
test_losses = [baseline_test_loss, modified_test_loss]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Accuracy bar chart
bars1 = axes[0].bar(models, test_accuracies, color=['steelblue', 'coral'], edgecolor='black', width=0.5)
axes[0].set_title('Test Accuracy Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_ylim(0.95, 1.0)  # Zoom in to see differences
for bar, acc in zip(bars1, test_accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f'{acc:.4f}', ha='center', fontsize=12, fontweight='bold')

# Loss bar chart
bars2 = axes[1].bar(models, test_losses, color=['steelblue', 'coral'], edgecolor='black', width=0.5)
axes[1].set_title('Test Loss Comparison', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Loss', fontsize=12)
for bar, loss in zip(bars2, test_losses):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f'{loss:.4f}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

### Experiment Discussion

**1. What was changed?**  
The number of neurons in the hidden layer was increased from 128 to 256. All other conditions (optimizer, loss function, learning rate, batch size, epochs, dataset, and preprocessing) were kept identical.

**2. What happened to performance?**  
The comparison table and plots above show the actual difference in test accuracy and loss between the two models. Refer to the printed values to see whether the modified model performed better, worse, or similarly.

**3. Why might increasing hidden neurons affect performance?**  
- **More neurons = more capacity:** With 256 neurons instead of 128, the hidden layer can learn more complex patterns and representations from the pixel data.
- **More parameters:** The modified model has significantly more trainable parameters, which could help capture finer distinctions between similar digits.

**4. Important caveat:**  
A larger model does **not** automatically guarantee better generalization. More parameters increase the risk of **overfitting**, where the model memorizes the training data instead of learning general patterns. This is why we monitor validation metrics. In practice, techniques like dropout and regularization help mitigate overfitting in larger models.

---

## Key Findings

Based on the actual results from this notebook:

In [ ]:
# ============================================================
# Generate Key Findings Programmatically from Actual Results
# ============================================================

# Calculate overfitting gap
baseline_overfit_gap = baseline_history.history['accuracy'][-1] - baseline_history.history['val_accuracy'][-1]
modified_overfit_gap = modified_history.history['accuracy'][-1] - modified_history.history['val_accuracy'][-1]

# Find most confused pair
max_confusion_idx = np.argmax(cm_off_diag)
most_confused_true = max_confusion_idx // 10
most_confused_pred = max_confusion_idx % 10
most_confused_count = cm_off_diag[most_confused_true, most_confused_pred]

# Accuracy difference
acc_diff = modified_test_accuracy - baseline_test_accuracy

print("===== KEY FINDINGS =====")
print()
print(f"1. The MNIST dataset consists of 70,000 grayscale images of handwritten digits (0–9),")
print(f"   each of size 28×28 pixels, belonging to 10 classes.")
print()
print(f"2. Normalizing pixel values from [0, 255] to [0, 1] is essential for stable and")
print(f"   efficient training. Without normalization, large input values slow convergence.")
print()
print(f"3. The baseline model (784 → 128 → 10) achieved a test accuracy of")
print(f"   {baseline_test_accuracy*100:.2f}% with a test loss of {baseline_test_loss:.4f}.")
print()
print(f"4. The modified model (784 → 256 → 10) achieved a test accuracy of")
print(f"   {modified_test_accuracy*100:.2f}% with a test loss of {modified_test_loss:.4f}.")
print(f"   Accuracy difference: {acc_diff*100:+.2f} percentage points.")
print()
print(f"5. From the confusion matrix, the most frequently confused pair was:")
print(f"   True digit {most_confused_true} misclassified as digit {most_confused_pred}")
print(f"   ({most_confused_count} times out of {len(y_test)} test images).")
print()
print(f"6. Overfitting gap (training accuracy - validation accuracy):")
print(f"   Baseline: {baseline_overfit_gap:.4f} | Modified: {modified_overfit_gap:.4f}")
if modified_overfit_gap > baseline_overfit_gap:
    print(f"   The modified model shows a slightly larger gap, suggesting marginally more overfitting.")
else:
    print(f"   Both models show similar overfitting behavior.")
print()
print(f"7. Even a simple feed-forward neural network with one hidden layer can achieve")
print(f"   high accuracy on MNIST, demonstrating that the dataset is well-suited for")
print(f"   introductory machine learning experiments.")

---

## Limitations

1. **Spatial information is lost:** By flattening the 28×28 image into a 784-element vector, the fully connected network treats each pixel independently and ignores the spatial relationships between neighboring pixels. A CNN would preserve this spatial structure.

2. **Results may vary slightly:** Due to the stochastic nature of weight initialization, mini-batch shuffling, and potential GPU non-determinism, results may differ slightly across different runs even with the same random seeds.

3. **Limited experimentation:** Only one model modification was tested (increasing hidden neurons from 128 to 256). Many other hyperparameters (learning rate, number of layers, dropout rate, batch size, etc.) could also be explored to better understand model behavior.

---

## Future Improvements

1. **Convolutional Neural Network (CNN):** Using convolutional layers would allow the model to capture spatial patterns (edges, curves, shapes) in the images, likely improving accuracy further.

2. **Regularization (Dropout):** Adding dropout layers between dense layers can reduce overfitting by randomly deactivating a fraction of neurons during each training step.

3. **Hyperparameter Tuning:** Systematically searching for the best learning rate, batch size, number of layers, and neurons per layer could yield better performance.

4. **Learning Rate Scheduling:** Reducing the learning rate during training (e.g., using learning rate decay or a scheduler) can help the model converge to a better solution.

5. **Data Augmentation:** Applying transformations like rotation, shifting, or scaling to training images can help the model generalize better to varied handwriting styles.

---

## Conclusion

In this notebook, we successfully built, trained, and evaluated a simple feed-forward neural network for the MNIST handwritten digit classification task.

**Key takeaways:**

- The **MNIST dataset** is a widely-used benchmark consisting of 70,000 grayscale images of digits 0–9, split into 60,000 training and 10,000 test images.

- **Preprocessing** is critical: normalizing pixel values to [0, 1] ensures stable training, and flattening images to 784-element vectors is necessary for a fully connected architecture.

- A **simple ANN** with architecture 784 → 128 → 10 can achieve strong performance on MNIST, demonstrating the power of even basic neural networks.

- **ReLU** introduces non-linearity in hidden layers, enabling the network to learn complex patterns, while **Softmax** converts output values into class probabilities for multi-class classification.

- **Training curves** help monitor learning progress and detect overfitting by comparing training and validation metrics across epochs.

- **Evaluation metrics** such as accuracy, confusion matrix, precision, recall, and F1-score provide a comprehensive view of model performance across all digit classes.

- **Experimentation** (increasing hidden neurons from 128 to 256) showed that model capacity can affect performance, but a larger model does not always guarantee better generalization.

This assignment demonstrates the fundamental workflow of a machine learning project: data loading, preprocessing, model building, training, evaluation, and experimentation — all essential skills for understanding and applying neural networks.

---

*End of Notebook*